In [ ]:
# 处理大型问卷时，建议写一个 Processor 类，将编码逻辑抽象化，避免手动重复操作

In [ ]:
import pandas as pd
import numpy as np

class SurveyProcessor:
    def __init__(self, file_path):
        """初始化：加载数据"""
        self.df = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)
        print(f"成功加载数据，原始维度: {self.df.shape}")

    def rename_columns(self, mapping_dict):
        """1. 重命名表头 (长问题 -> 短变量名)"""
        self.df.rename(columns=mapping_dict, inplace=True)
        return self

    def filter_invalid(self, min_duration=10):
        """2. 过滤无效样本 (用时太短或未完成)"""
        if 'duration' in self.df.columns:
            self.df = self.df[self.df['duration'] >= min_duration]
            
        print(f"过滤后维度: {self.df.shape}")
        return self

    def handle_missing(self, strategy_dict): 
        """3. 处理缺失值 (自定义每一列的填充方式)
        {'age': 'mean', 'city': 'mode', 'score': 0}"""
        for col, strategy in strategy_dict.items():
            if col in self.df.columns:
                if strategy == 'mean':
                    self.df[col] = self.df[col].fillna(self.df[col].mean())
                elif strategy == 'mode': #mode返回众数序列
                    self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
                else: #指定默认值
                    self.df[col] = self.df[col].fillna(strategy)
        return self

    def encode_ordinal(self, col_mappings):
        """4. 定序编码 (Likert量表)"""
        for col, mapping in col_mappings.items():
            if col in self.df.columns:
                self.df[col] = self.df[col].map(mapping)
        return self

    def encode_nominal(self, columns):
        """5. 定类编码 (One-Hot)"""
        self.df = pd.get_dummies(self.df, columns=columns, prefix=columns)
        return self

    def encode_multi_select(self, col_list, sep=';'):
        """6. 多选题编码 (拆分字符串并二值化)"""
        for col in col_list:
            if col in self.df.columns:
                # 转化为 0/1 矩阵
                dummies = self.df[col].str.get_dummies(sep=sep).add_prefix(f"{col}_")
                self.df = pd.concat([self.df, dummies], axis=1)
                self.df.drop(columns=[col], inplace=True) # 删除原多选题列
        return self

    def extract_numeric(self, columns):
        """7. 提取数值 (例如 '25岁' -> 25)"""
        for col in columns:
            self.df[col] = self.df[col].astype(str).str.extract(r'(\d+)').astype(float)
            #\d：代表匹配任何数字（0-9） +：代表匹配一个或多个连续的数字 ()：代表“捕获组”
        return self

    def get_data(self):
        """最终返回 DataFrame"""
        return self.df

In [ ]:
# my_project/
# ├── main.py                
# └── survey_processor.py    # 存放 SurveyProcessor 类的地方
from survey_processor import SurveyProcessor

# my_project/
# ├── main.py
# └── utils/
#     └── survey_processor.py  # 在子文件夹里
# 添加 __init__.py：在 utils 文件夹里创建一个空的 __init__.py 文件，把该文件夹变成一个 Python 包
from utils.survey_processor import SurveyProcessor

In [ ]:
from survey_processor import SurveyProcessor

# 1. 定义配置（把配置和逻辑分离，是程序员的好习惯）
COLUMN_MAP = {
    '您目前的年龄': 'age',
    '您的职业': 'job',
    '您对产品的整体满意度': 'satisfaction',
    '您平时使用的功能（多选）': 'features',
    '您所在的城市': 'city'
}

SATISFACTION_MAP = {
    '非常不满意': 1, '不满意': 2, '一般': 3, '满意': 4, '非常满意': 5
}

# 2. 链式调用处理流程
processor = SurveyProcessor('raw_data.csv')

clean_df = (processor
            .rename_columns(COLUMN_MAP)
            .filter_invalid(min_duration=30) # 过滤答题少于30秒的
            .extract_numeric(['age'])         # 处理年龄列
            .encode_ordinal({'satisfaction': SATISFACTION_MAP}) # 处理满意度
            .encode_nominal(['city'])        # 城市做独热编码
            .encode_multi_select(['features']) # 拆解多选题
            .handle_missing({'job': 'Unknown', 'age': 'mean'}) # 填充缺失值
            .get_data())

# 3. 查看结果
print(clean_df.head())
clean_df.to_csv('cleaned_data.csv', index=False)